## # 🤖 Stage 1: Foundational Models & Basic Invocation

Welcome to the first module! In this notebook, we establish the fundamental building blocks of interacting with Large Language Models (LLMs) using LangChain. 

Before building complex agents, it is crucial to understand how to securely load environment variables, initialize a foundational chat model, and properly parse the objects returned by the model.

## Initialising and invoking a model

### 1. Environment Setup
Security first. We use the `python-dotenv` library to load our API keys from a hidden `.env` file. This prevents hardcoding sensitive credentials directly into the source code, which is a critical best practice for production environments.

In [1]:
from dotenv import load_dotenv

load_dotenv()

True

### 2. Initializing the Foundation Model
LangChain provides a unified interface to interact with various model providers. By using `init_chat_model`, we instantiate our foundational model. This abstraction allows us to seamlessly swap out models (e.g., switching from OpenAI to Anthropic) later in production without rewriting our core logic.

In [2]:
from langchain.chat_models import init_chat_model
model= init_chat_model("gpt-5-nano") 

### 3. Invoking the Model
We send our prompt to the model using the `.invoke()` method. 

**Note on Output:** The LLM does not return a simple raw string. Instead, it returns a structured `AIMessage` object. This object contains not only the text response but also valuable metadata (like token usage and stop reasons) which is essential for debugging and cost tracking.

In [3]:
response= model.invoke("what is the capital of the moon?")
response

AIMessage(content='There isn’t one. The Moon isn’t a country and has no government or capital.\n\nIf you meant something else:\n- Capital letter: The capital (uppercase) letter of the word "moon" is M.\n- Fiction: Some sci-fi stories imagine a lunar capital with names like Lunapolis or Luna City, but that’s fictional.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 846, 'prompt_tokens': 14, 'total_tokens': 860, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 768, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DpMKhjpTPfNsAG5EyHTsSJU9Ci8Ra', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019eb3cb-ee96-7a71-8355-0ef58425547a-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_to

### 4. Parsing the Output
To extract the actual human-readable text generated by the model, we access the `.content` attribute of the `AIMessage` object.

In [4]:
response.content

'There isn’t one. The Moon isn’t a country and has no government or capital.\n\nIf you meant something else:\n- Capital letter: The capital (uppercase) letter of the word "moon" is M.\n- Fiction: Some sci-fi stories imagine a lunar capital with names like Lunapolis or Luna City, but that’s fictional.'

### 5. Inspecting Model Metadata
Beyond the raw text generation, the LLM returns a comprehensive metadata dictionary. This includes critical information such as token usage (`prompt_tokens`, `completion_tokens`), finish reasons, and the exact model version used. Inspecting this data is essential for monitoring API costs and debugging truncation issues in production.

In [5]:
from pprint import pprint
pprint(response.response_metadata)

{'finish_reason': 'stop',
 'id': 'chatcmpl-DpMKhjpTPfNsAG5EyHTsSJU9Ci8Ra',
 'logprobs': None,
 'model_name': 'gpt-5-nano-2025-08-07',
 'model_provider': 'openai',
 'service_tier': 'default',
 'system_fingerprint': None,
 'token_usage': {'completion_tokens': 846,
                 'completion_tokens_details': {'accepted_prediction_tokens': 0,
                                               'audio_tokens': 0,
                                               'reasoning_tokens': 768,
                                               'rejected_prediction_tokens': 0},
                 'prompt_tokens': 14,
                 'prompt_tokens_details': {'audio_tokens': 0,
                                           'cached_tokens': 0},
                 'total_tokens': 860}}


## Customising your Model

### 6. Customizing Model Parameters (Temperature)
We can control the LLM's behavior by passing specific parameters during initialization. Here, we adjust the `temperature` parameter to `1.0`. A higher temperature increases randomness and creativity, which is ideal for brainstorming, creative writing, or generating fictional content (like creating a fictional capital of the moon).

In [6]:
model= init_chat_model("gpt-5-nano",
                       temperature= 1.0)
response= model.invoke("what is the capital of the moon?")

print(response.content)

There isn’t one. The Moon isn’t a country or governing body, so it has no capital.

If you meant it as a riddle: the capital letter of the word "Moon" is M.

If you meant its name in mythology, common names are Luna (Latin) or Selene (Greek), but that’s not a capital either.


## Initialising and invoking an agent

---
## 🛠️ Agents and Message Standardization
### 1. Initializing an Agent and Standard Messages
While a foundational model simply processes strings, an **Agent** in LangChain is designed to manage state, execute tools, and handle complex logic. To communicate effectively with agents, we use LangChain's standardized message classes (such as `HumanMessage` and `AIMessage`) rather than raw text strings.

In [7]:
from langchain.agents import create_agent

agent= create_agent("gpt-5-nano")

In [8]:
from langchain.messages import HumanMessage

response= agent.invoke({"messages": [HumanMessage(content= "what is the capital of the moon?")]})

In [9]:
from pprint import pprint
pprint(response)

{'messages': [HumanMessage(content='what is the capital of the moon?', additional_kwargs={}, response_metadata={}, id='7cb0aa91-98fd-42ee-b8d4-aa3722b004e3'),
              AIMessage(content='There isn’t one. The Moon isn’t a country or government, so it has no capital city. Under current international law (Outer Space Treaty), no nation owns the Moon and there’s no formal lunar government.\n\nIf you’re thinking in fiction, you might encounter invented capitals like Lunapolis or Moon City, but those are purely fictional. If you want, I can share examples from specific books or games.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 1051, 'prompt_tokens': 14, 'total_tokens': 1065, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 960, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-nano-202

In [10]:
print(response["messages"][-1].content)

There isn’t one. The Moon isn’t a country or government, so it has no capital city. Under current international law (Outer Space Treaty), no nation owns the Moon and there’s no formal lunar government.

If you’re thinking in fiction, you might encounter invented capitals like Lunapolis or Moon City, but those are purely fictional. If you want, I can share examples from specific books or games.


### 2. Multi-turn Conversations (Context Memory)
LLMs are inherently stateless; they do not remember previous interactions. To maintain a conversation, we must pass the entire history of the dialogue as a list of messages. The agent analyzes the preceding `HumanMessage` and `AIMessage` objects to understand the context before generating its next response.

In [11]:
from langchain.messages import AIMessage

response= agent.invoke({"messages": [HumanMessage(content= "what is the capital of the moon?"   ),
                                      AIMessage(content= "the capital of the moon is luna city.") ,
                                      HumanMessage(content= "Interesting, tell me more about Luna City")]})

pprint(response)

{'messages': [HumanMessage(content='what is the capital of the moon?', additional_kwargs={}, response_metadata={}, id='ff5417fb-7d0c-4246-b752-9092389d0c58'),
              AIMessage(content='the capital of the moon is luna city.', additional_kwargs={}, response_metadata={}, id='c6100cfb-9c82-42f7-9893-41d6d0f3ccda', tool_calls=[], invalid_tool_calls=[]),
              HumanMessage(content='Interesting, tell me more about Luna City', additional_kwargs={}, response_metadata={}, id='1f25164c-ddbf-44f7-aeb7-d8cb7e407541'),
              AIMessage(content="Since Luna City is a fictional setting, here’s a richer, believable portrait you can use for stories or world-building.\n\nQuick snapshot\n- Luna City sits inside the edge of a large crater on the Moon, a capital of governance and culture built in fusion with the lunar geology. Glass-domed neighborhoods rise beside shielded lava-tube corridors, powered by solar farms and a network of water and energy infrastructure harvested from the Moo

## Streaming Output

---
## 🌊 Streaming Output
For long or complex responses, waiting for the entire generation to complete before displaying it causes a poor User Experience (UX). By using the `.stream()` method, we can yield and print tokens one by one as they are generated by the model, creating a real-time typewriter effect.

In [13]:
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="Tell me all about Luna City, the capital of the Moon")]},
    stream_mode="messages"
):
    if token.content:
        print(token.content, end="", flush=True)

Luna City is a fictional concept you can use for a story, game, or worldbuilding setting. It doesn’t exist in real life, but you can build a rich, plausible capital for the Moon around it. Below is a comprehensive, self-contained dossier you can copy, adapt, or expand.

Overview
- Core idea: Luna City is the Moon’s political and cultural capital, a sprawling, multi-layered settlement that blends advanced lunar engineering with a diverse, planet-spanning population. It sits at the crossroads of science, governance, commerce, and culture, and serves as the orbital and interplanetary hub for Earth’s lunar activities.
- Tone options: Hard sci-fi plausibility (tightly engineered, realistic constraints) or more speculative/futuristic (more exotic tech and social systems). I’ll give you a solid hard-sci-fi baseline with room to add cinematic elements.

Quick facts (at-a-glance)
- Location: Near the Moon’s equator on the near side, within a large crater rim and adjoining crater floors to maxim

## Basic prompting

---
## 🎭 Prompt Engineering
### 1. System Prompts (Persona Assignment)
A System Prompt acts as the overarching instruction manual for the LLM. By assigning a specific persona (e.g., "You are a science fiction writer"), we drastically alter the tone, vocabulary, and style of the output without modifying the user's core request.

In [14]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

agent= create_agent("gpt-5-nano")  

question= HumanMessage(content= "What is the capital of the moon?")

response= agent.invoke({"messages":[question]})

print(response["messages"][1].content)

There isn’t one. The Moon isn’t a country or a government, so it has no capital city.

If you’re asking in a playful or fictional sense, some sci‑fi stories imagine a capital like “Luna City” or “Lunapolis,” but those are not real. If you want, tell me a book or movie and I can check what they call it there.


In [15]:
system_prompt="You are a science fiction writer, create a capital city at the users request."

scifi_agent= create_agent("gpt-5-nano", system_prompt= system_prompt)

response= scifi_agent.invoke({"messages":[question]})

print(response["messages"][1].content)

In this sci‑fi universe, the Moon’s capital is called Lunaris Prime, sometimes lovingly known as Selenopolis by poets and settlers.

Key details:
- Location: A grand city built into and around the rim of Tycho Crater, on the Moon’s near side for easy Earth contact. The urban core sprawls along terraces cut into the crater wall, with networks of glassy corridors linking to subterranean lava-tube habitats.
- Government: The Lunar Conclave, a representative council elected from district cores, with a rotating “First Luminary” who oversees interplanetary relations and lunar defense.
- Population: Several million residents live in the domed districts and underground habitats, with a seasonal influx of scientists and tourists.
- Architecture and life: White basalt façades, translucent regolith domes, and solar-reflective towers. Parks float in microgravity corridors, and magnetized trams drift along magnetic rails. Water is recycled, energy is drawn from pristine heliostat fields, and art in

## Few-shot examples

### 2. Few-Shot Prompting
Sometimes, explicit instructions are not enough to guarantee a specific format. By providing a few examples of the desired input-output pairs directly within the system prompt, we effectively "train" the model in-context. This significantly improves consistency and predictable formatting.

In [17]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

User: What is the capital of mars?
Scifi Writer: Marsialis

User: What is the capital of Venus?
Scifi Writer: Venusovia

"""

scifi_agent= create_agent("gpt-5-nano", system_prompt= system_prompt)

response= scifi_agent.invoke({"messages":[question]})

print(response["messages"][1].content)

Lunapolis


## Structured prompts

### 3. Structured Text Prompts
We can enforce a specific layout by explicitly defining the required layout keys (Name, Location, Vibe, Economy) in the system prompt. While this organizes the text output, it still returns a plain string, which requires complex Regex or parsing to extract programmatically.

In [18]:
system_prompt = """

You are a science fiction writer, create a space capital city at the users request.

Please keep to the below structure.

Name: The name of the capital city

Location: Where it is based

Vibe: 2-3 words to describe its vibe

Economy: Main industries

"""

scifi_agent= create_agent("gpt-5-nano", 
                          system_prompt= system_prompt)

response= scifi_agent.invoke({"messages":[question]})

print(response["messages"][1].content)

Name: Lunaris Prime

Location: Near-side of the Moon, Mare Tranquillitatis region, at the edge of a shielded crater in the equatorial belt.

Vibe: Futuristic frontier

Economy: Helium-3 mining, ISRU-based manufacturing, lunar construction, space tourism, and research & development


## Structured output

---
## 🏗️ Structured Output (Pydantic)
### Guaranteed Object Structures for Production
Relying on plain text formatting is brittle. By defining a Pydantic `BaseModel` (`CapitalInfo`) and passing it as the `response_format` to our agent, we force the LLM to return a strictly typed, predictable object instead of a string. This is the industry standard for 100% reliable data extraction and API integration.

In [21]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage
from pydantic import BaseModel

class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

scifi_agent= create_agent("gpt-5-nano", 
                          system_prompt= "You are a science fiction writer, create a capital city at the users request.",
                              response_format= CapitalInfo)

response= scifi_agent.invoke({"messages":[question]})  

print(response["structured_response"])

name='Selene Prime' location='Shackleton Crater rim, south polar region of the Moon' vibe='A shimmering, solar-powered metropolis carved into ice and glass, where governance, science, and culture meet in lunar daylight. Neon domes, crystal transit, and artful architecture thrive in microgravity.' economy='Helium-3 mining, water-ice harvesting, in-situ resource utilization, lunar research and education, and spaceport commerce.'


### Accessing Structured Data
Since the model's output is now a parsed Python object rather than raw text, we can seamlessly access individual attributes (like `.name` or `.location`) programmatically using standard dot notation.

In [22]:
response["structured_response"].name

'Selene Prime'

In [23]:
capital_info = response["structured_response"]

capital_name = capital_info.name
capital_location = capital_info.location

print(f"{capital_name} is a city located at {capital_location}")

Selene Prime is a city located at Shackleton Crater rim, south polar region of the Moon
